# Andrea Pezzolla & Giovanni Russo


In [1]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
# Dataframe
df = pd.read_csv("data.csv", sep=';')

# Dati utili per tutti i plot
num_anni = df['Unfalljahr'].nunique()


In [ ]:
# Dizionari per la traduzione tedesco-italiano
traduzione_giorni = {
    '1 Montag': '1 Lunedì', '2 Dienstag': '2 Martedì', '3 Mittwoch': '3 Mercoledì',
    '4 Donnerstag': '4 Giovedì', '5 Freitag': '5 Venerdì', '6 Samstag': '6 Sabato', '7 Sonntag': '7 Domenica'
}

traduzione_gravita = {
    '1 Unfall mit Sachschaden': '1 Solo danni materiali',
    '2 Unfall mit Leichtverletzten': '2 Feriti lievi',
    '3 Unfall mit Schwerverletzten': '3 Feriti gravi',
    '4 Unfall mit Getöteten': '4 Mortali'
}

traduzione_strade = {
    'Hauptstrasse': 'Strada Principale',
    'Nebenstrasse': 'Strada Secondaria',
    'Autobahn': 'Autostrada',
    'andere': 'Altro'
}

# Applichiamo traduzioni al Dataframe
df['Wochentag_IT'] = df['Wochentag'].map(traduzione_giorni)
df['Gravita_IT'] = df['Beschreibung der Unfallschwerekategorie'].map(traduzione_gravita)
df['Strassenart_IT'] = df['Strassenart'].map(traduzione_strade).fillna(df['Strassenart'])

# Dizionario i campi
label_it = {
    'Unfallstunde': 'Ora del giorno',
    'Wochentag_IT': 'Giorno della settimana',
    'Gravita_IT': 'Gravità',
    'Unfalljahr': 'Anno',
    'Strassenart_IT': 'Tipo di strada',
    'Totale': 'Numero di Incidenti',
    'Lat': 'Latitudine', 'Lon': 'Longitudine'
}

# Dizionario utenti
traduzione_utenti = {
    'Fahrradbeteiligung': 'Bicicletta',
    'Motorradbeteiligung': 'Moto',
    'Fussgängerbeteiligung': 'Pedoni'
}

# Mappa colori
color_map = {
    '1 Solo danni materiali': 'green',
    '2 Feriti lievi': 'gold',
    '3 Feriti gravi': 'red',
    '4 Mortali': 'black'
}


In [4]:
# HeatMap completa

# Divido il df per avere il conteggio all'anno e non totale
heat_df = (
    df.groupby(['Wochentag_IT', 'Unfallstunde', 'Unfalljahr'])
    .size()
    .reset_index(name='conteggio')
    .groupby(['Wochentag_IT', 'Unfallstunde'])['conteggio']
    .mean()
    .reset_index(name='media_annua')
)
heat_df['media_annua'] = heat_df['media_annua'].round(1)

# Ordine dei giorni
sorted_days_it = [
    '1 Lunedì', '2 Martedì', '3 Mercoledì',
    '4 Giovedì', '5 Venerdì', '6 Sabato', '7 Domenica'
]

# HeatMap
fig = px.density_heatmap(
    heat_df,
    x='Unfallstunde',
    y='Wochentag_IT',
    z='media_annua',
    histfunc='avg',
    title="Mappa di Calore: Media annua degli incidenti per Giorno e Ora",
    labels={**label_it,},
    color_continuous_scale='Inferno',
    category_orders={'Wochentag_IT': sorted_days_it},
    nbinsx=24
)

# Metto 1 per avere tutti e 24 le ore scritte sotto
fig.update_layout(
    xaxis=dict(dtick=1),
    coloraxis_colorbar=dict(title="Media annua incidenti")
)

fig.show()

In [5]:
# HeatMap solo gravi e morti

# Prendo solo gravi e morti
df_gravi = df[df['Gravita_IT'].isin(['3 Feriti gravi', '4 Mortali'])]

# Divido il df per avere il conteggio all'anno e non totale
heat_df_gravi = (
    df_gravi.groupby(['Wochentag_IT', 'Unfallstunde', 'Unfalljahr'])
    .size()
    .reset_index(name='conteggio')
    .groupby(['Wochentag_IT', 'Unfallstunde'])['conteggio']
    .mean()
    .reset_index(name='media_annua')
)
heat_df_gravi['media_annua'] = heat_df_gravi['media_annua'].round(1)

# HeatMap
fig_gravi = px.density_heatmap(
    heat_df_gravi,
    x='Unfallstunde',
    y='Wochentag_IT',
    z='media_annua',
    histfunc='avg',
    title="Mappa di Calore: Media annua incidenti gravi e mortali per Giorno e Ora",
    labels={**label_it, 'media_annua': 'Media annua incidenti'},
    color_continuous_scale='Inferno',
    category_orders={'Wochentag_IT': sorted_days_it},
    nbinsx=24
)

# Metto 1 per avere tutti e 24 le ore scritte sotto
fig_gravi.update_layout(
    xaxis=dict(dtick=1),
    coloraxis_colorbar=dict(title="Incidenti/anno")
)

fig_gravi.show()

In [6]:
# Grafico a torta

# Calcolo delle medie annuali per ogni categoria
conteggi_gravita = df['Gravita_IT'].value_counts().reset_index()
conteggi_gravita.columns = ['Gravita_IT', 'Totale']
conteggi_gravita['Media_Annuale'] = (conteggi_gravita['Totale'] / num_anni).round(0).astype(int)
conteggi_gravita['Etichetta_Legenda'] = conteggi_gravita['Gravita_IT'] + " (Circa: " + conteggi_gravita['Media_Annuale'].astype(str) + "/anno)"

# Pie
color_map_pie = {row['Etichetta_Legenda']: color_map[row['Gravita_IT']] for _, row in conteggi_gravita.iterrows()}

fig1 = px.pie(
    conteggi_gravita, 
    names='Etichetta_Legenda', 
    values='Totale',
    color='Etichetta_Legenda',
    color_discrete_map=color_map_pie,
    title=f"Distribuzione Gravità Incidenti (Media su {num_anni} anni)"
)


fig1.update_traces(
    textposition='inside', 
    hovertemplate="<b>%{label}</b><br>Totale incidenti: %{value}<extra></extra>"
)

fig1.show()

In [7]:
# ScatterMap

# Prendo le coordinate
df[['Lat', 'Lon']] = df['Geo Point'].str.split(', ', expand=True).astype(float)

# Dimensioni per gravità
size_map = {
    '1 Solo danni materiali': 1,
    '2 Feriti lievi': 2,
    '3 Feriti gravi': 3,
    '4 Mortali': 4
}
df['Dimensione'] = df['Gravita_IT'].map(size_map)

#ScatterMap

fig2 = px.scatter_map(
    df, 
    lat="Lat", 
    lon="Lon", 
    color="Gravita_IT",
    color_discrete_map=color_map,
    size="Dimensione",
    size_max=10,
    hover_name="Gravita_IT",
    hover_data={"Lat": False, "Lon": False, "Unfalljahr": True, "Strassenart_IT": True, "Gravita_IT": False, "Dimensione": False},
    title="Mappa degli incidenti per Gravità",
    labels=label_it, 
    map_style="carto-positron",
    zoom=11,
    opacity=0.7
)

fig2.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig2.show()

In [8]:
# Subplot tipi di strada e gravità

# Rimuovo Area adiacente perché pochi dati
df_filtered = df[~df['Strassenart_IT'].isin(['Area adiacente', 'Nebenlage', 'Nebenanlage'])]

# Definisco in che ordine voglio tutto
ordine_gravita = ['1 Solo danni materiali', '2 Feriti lievi', '3 Feriti gravi', '4 Mortali']
ordine_gravita_sinistra = ordine_gravita[::-1]
ordine_strade = sorted(df_filtered['Strassenart_IT'].unique(), reverse=True)

# Aggrego per avere media annua
df_road = (
    df_filtered.groupby(['Strassenart_IT', 'Gravita_IT'])
    .size()
    .reset_index(name='Totale')
)
df_road['Media_Annua'] = df_road['Totale'] / num_anni

# Se non ho valori per una combinazione la metto a 0
multi_idx = pd.MultiIndex.from_product(
    [ordine_strade, ordine_gravita],
    names=['Strassenart_IT', 'Gravita_IT']
)
df_road = (
    df_road
    .set_index(['Strassenart_IT', 'Gravita_IT'])
    .reindex(multi_idx, fill_value=0)
    .reset_index()
)

# Salvo la percentuale per ogni strada
totale_per_strada = df_road.groupby('Strassenart_IT')['Media_Annua'].transform('sum')
df_road['Percentuale'] = (df_road['Media_Annua'] / totale_per_strada * 100).fillna(0)

# Definisco l'ordine usando le maschere sopra
df_road['Strassenart_IT'] = pd.Categorical(df_road['Strassenart_IT'], categories=ordine_strade, ordered=True)
df_road = df_road.sort_values(['Strassenart_IT', 'Gravita_IT'])

# Definisco la struttura del SubPlot
fig3_road = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Media annua incidenti", "Distribuzione percentuale"),
    shared_yaxes=True,
    column_widths=[0.65, 0.35]
)

# Grafico a sinistra
for gravita in ordine_gravita_sinistra:
    subset = df_road[df_road['Gravita_IT'] == gravita]
    colore = color_map[gravita]

    fig3_road.add_trace(
        go.Bar(
            x=subset['Media_Annua'],
            y=subset['Strassenart_IT'],
            name=gravita,
            orientation='h',
            marker_color=colore,
            legendgroup=gravita,
            offsetgroup=gravita,   # <-- aggiunto
            hovertemplate='%{x:.1f} incidenti/anno<extra>' + gravita + '</extra>'
        ),
        row=1, col=1
    )

# Grafico a destra
cumsum = {strada: 0.0 for strada in ordine_strade}

for gravita in ordine_gravita:
    subset = (
        df_road[df_road['Gravita_IT'] == gravita]
        .set_index('Strassenart_IT')
    )

    x_vals    = [subset.loc[s, 'Percentuale'] if s in subset.index else 0.0 for s in ordine_strade]
    base_vals = [cumsum[s] for s in ordine_strade]

    for s, x in zip(ordine_strade, x_vals):
        cumsum[s] += x

    fig3_road.add_trace(
        go.Bar(
            x=x_vals,
            y=ordine_strade,
            base=base_vals,
            name=gravita,
            orientation='h',
            marker_color=color_map[gravita],
            legendgroup=gravita,
            showlegend=False,
            width=0.6,
            offsetgroup='right', 
            hovertemplate='%{x:.1f}%<extra>' + gravita + '</extra>'
        ),
        row=1, col=2
    )

# Layout
fig3_road.update_layout(
    barmode='group',
    title="Tipo di Strada e Gravità — media annua e distribuzione %",
    legend_title="Gravità",
    legend_traceorder='reversed', 
    height=420
)

fig3_road.update_xaxes(title_text="Incidenti/anno", row=1, col=1)
fig3_road.update_xaxes(title_text="Percentuale (%)", range=[0, 100], row=1, col=2)
fig3_road.update_yaxes(
    categoryorder='array',
    categoryarray=ordine_strade,
    row=1, col=1
)

fig3_road.show()

In [9]:
# SubPlot utente e gravità

# Prendo solo gli incidenti con moto bici e pedoni
df_melt_utenti = df.melt(
    id_vars=['Gravita_IT', 'Unfalljahr'], 
    value_vars=['Fahrradbeteiligung', 'Motorradbeteiligung', 'Fussgängerbeteiligung'],
    var_name='Utente', 
    value_name='Coinvolto'
)
df_coinvolti = df_melt_utenti[df_melt_utenti['Coinvolto'] == True]
df_coinvolti = df_coinvolti.copy()
df_coinvolti['Utente_IT'] = df_coinvolti['Utente'].map(traduzione_utenti)

# Definisco in che ordine voglio tutto
ordine_gravita = ['1 Solo danni materiali', '2 Feriti lievi', '3 Feriti gravi', '4 Mortali']
ordine_gravita_sinistra = ordine_gravita[::-1]
ordine_utenti = ['Bicicletta', 'Moto', 'Pedoni']

# Aggrego per avere la media annua
df_utenti = (
    df_coinvolti.groupby(['Utente_IT', 'Gravita_IT'])
    .size()
    .reset_index(name='Totale')
)
df_utenti['Media_Annua'] = df_utenti['Totale'] / num_anni

# Se non ho valori per una combinazione la metto a 0
multi_idx = pd.MultiIndex.from_product(
    [ordine_utenti, ordine_gravita],
    names=['Utente_IT', 'Gravita_IT']
)
df_utenti = (
    df_utenti
    .set_index(['Utente_IT', 'Gravita_IT'])
    .reindex(multi_idx, fill_value=0)
    .reset_index()
)

# Salvo la percentuale per ogni tipo di utente
totale_per_utente = df_utenti.groupby('Utente_IT')['Media_Annua'].transform('sum')
df_utenti['Percentuale'] = (df_utenti['Media_Annua'] / totale_per_utente * 100).fillna(0)

# Definisco l'ordine usando la lista ordine_utenti
df_utenti['Utente_IT'] = pd.Categorical(df_utenti['Utente_IT'], categories=ordine_utenti, ordered=True)
df_utenti = df_utenti.sort_values(['Utente_IT', 'Gravita_IT'])

# Definisco la struttura del SubPlot
fig3_utenti = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Media annua incidenti", "Distribuzione percentuale"),
    shared_yaxes=True,
    column_widths=[0.65, 0.35]
)

# Grafico a sinistra
for gravita in ordine_gravita_sinistra:
    subset = df_utenti[df_utenti['Gravita_IT'] == gravita]
    colore = color_map[gravita]

    fig3_utenti.add_trace(
        go.Bar(
            x=subset['Media_Annua'],
            y=subset['Utente_IT'],
            name=gravita,
            orientation='h',
            marker_color=colore,
            legendgroup=gravita,
            offsetgroup=gravita,
            hovertemplate='%{x:.1f} incidenti/anno<extra>' + gravita + '</extra>'
        ),
        row=1, col=1
    )

# Grafico a destra: stack manuale tramite base
cumsum = {utente: 0.0 for utente in ordine_utenti}

for gravita in ordine_gravita:
    subset = (
        df_utenti[df_utenti['Gravita_IT'] == gravita]
        .set_index('Utente_IT')
    )

    x_vals    = [subset.loc[u, 'Percentuale'] if u in subset.index else 0.0 for u in ordine_utenti]
    base_vals = [cumsum[u] for u in ordine_utenti]

    for u, x in zip(ordine_utenti, x_vals):
        cumsum[u] += x

    fig3_utenti.add_trace(
        go.Bar(
            x=x_vals,
            y=ordine_utenti,
            base=base_vals,
            name=gravita,
            orientation='h',
            marker_color=color_map[gravita],
            legendgroup=gravita,
            offsetgroup='right',
            showlegend=False,
            width=0.6,
            hovertemplate='%{x:.1f}%<extra>' + gravita + '</extra>'
        ),
        row=1, col=2
    )

# SubPlot
fig3_utenti.update_layout(
    barmode='group',
    title="Gravità degli incidenti: Coinvolgimento Bici, Moto e Pedoni — media annua e distribuzione %",
    legend_title="Gravità",
    legend_traceorder='reversed',
    height=380
)

fig3_utenti.update_xaxes(title_text="Incidenti/anno", row=1, col=1)
fig3_utenti.update_xaxes(title_text="Percentuale (%)", range=[0, 100], row=1, col=2)
fig3_utenti.update_yaxes(
    categoryorder='array',
    categoryarray=ordine_utenti,
    row=1, col=1
)

fig3_utenti.show()

In [10]:
# Istogramma Auto / altro

# Faccio una maschera per solo auto
condizione_solo_auto = (df['Fahrradbeteiligung'] == False) & \
                       (df['Motorradbeteiligung'] == False) & \
                       (df['Fussgängerbeteiligung'] == False)

df['Tipologia_Coinvolti'] = 'Pedoni, Moto e Bici'
df.loc[condizione_solo_auto, 'Tipologia_Coinvolti'] = 'Solo auto'

# Definisco ordine
ordine_gravita = ['1 Solo danni materiali', '2 Feriti lievi', '3 Feriti gravi', '4 Mortali']

# Aggrego per avere la media annua
df_tipo = (
    df.groupby(['Tipologia_Coinvolti', 'Gravita_IT'])
    .size()
    .reset_index(name='Totale')
)
df_tipo['Media_Annua'] = df_tipo['Totale'] / num_anni

# Grafico a barre
fig3_tipo = px.bar(
    df_tipo,
    y="Tipologia_Coinvolti",
    x="Media_Annua",
    color="Gravita_IT",
    color_discrete_map=color_map,
    category_orders={"Gravita_IT": ordine_gravita},
    title="Gravità degli incidenti/anno: Solo Auto vs Pedoni, Moto e Bici",
    barmode="group",
    labels={
        "Tipologia_Coinvolti": "Utenti Coinvolti",
        "Media_Annua": "Incidenti/anno",
        "Gravita_IT": "Gravità"
    }
)

fig3_tipo.update_layout(xaxis_title="Incidenti/anno")
fig3_tipo.show()